Robert Armenta , Oscar Ramos

Decision Tree

Decision Tree Dataset: https://www.kaggle.com/datasets/saritanandal/cement-demand-forecasting-dataset

In [ ]:
import pandas as pd                  # Used for loading and manipulating datasets
import numpy as np                   # Used for numerical operations
import matplotlib.pyplot as plt     # Used for plotting graphs if needed
from sklearn.tree import DecisionTreeClassifier  # Decision Tree model
from sklearn.model_selection import train_test_split  # Used to split data into train/test
from sklearn import metrics         # Used to evaluate model performance
from sklearn.preprocessing import LabelEncoder   # Converts categorical labels to numbers



In [ ]:
# This lets you upload your CSV file directly into Colab

from google.colab import files
uploaded = files.upload()


In [ ]:
# Load the Dataset
# Read the cement dataset from the CSV file
df = pd.read_csv("Generated_cement_demand_dataset.csv")

# Display the first few rows of the dataset to understand the data structure
df.head()


In [ ]:
# Data Cleaning / Preprocessing
# Remove the "Type of Cements" column because it is not needed for prediction
df.drop(['Type of Cements'], axis=1, inplace=True)

# Check dataset again after dropping the column
df.head()

In [ ]:
# Date and Location are not numeric features useful for the model
# GDP Growth is removed to simplify the model
df.drop(['Date', 'Location', 'GDP Growth (%)'], axis=1, inplace=True)

# View the dataset again after cleaning
df.head()

In [ ]:
# Convert Categorical Target Variable to Numeric
# Create LabelEncoder object
le = LabelEncoder()

# Convert the Promotion column from text labels to numbers
# Example: No Promotion -> 0, Promotion -> 1
label = le.fit_transform(df["Promotion"])

# Replace original column with encoded numeric values
df["Promotion"] = label

# Display encoded labels
label

In [ ]:
# Define Features (X) and Target Variable (y)
# Features used to predict promotion
x_data = df[['Population',
             'Inventory Levels (tons)',
             'Transportation Costs',
             'Sales',
             'Price']]

# Target variable we want to predict
y_data = df['Promotion']

In [ ]:
# Split Dataset into Training and Testing Sets
# 70% of data will be used to train the model
# 30% will be used to test the model performance
X_train, X_test, y_train, y_test = train_test_split(
    x_data, y_data, test_size=0.3, random_state=1)


In [ ]:
# Create and Train the Decision Tree Model
dt_model = DecisionTreeClassifier()

# Train Decision Tree Classifer
dt_model2 = dt_model.fit(X_train,y_train)

#Predict the response for test dataset
y_pred = dt_model2.predict(X_test)


#Model Accuracy
print("Accuracy:",metrics.accuracy_score(y_test, y_pred))



In [ ]:
# Measure precision, recall, and f1 score.
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred))

precision = metrics.precision_score(y_test, y_pred)
recall = metrics.recall_score(y_test, y_pred)
f1 = metrics.f1_score(y_test, y_pred)

print(f"The precision of the model is {precision:.2f}.")
print(f"The recall of the model is {recall:.2f}.")
print(f"The F1-score of the model is {f1:.2f}.")



In [ ]:


# Function to Allow User Input for Predictions


def predict_cement_type():
    while True:      # Loop allows multiple predictions without restarting the program
        try:             # Ask the user to enter values for each feature
            population = float(input("Enter population: "))
            inventory_levels = float(input("Enter inventory levels (tons): "))
            transport_costs = float(input("Enter transportation costs: "))
            price = float(input("Enter price: "))
            sales = float(input("Enter sales: "))

            if any(val < 0 for val in [    #Check if user entered negative numbers
                      population, inventory_levels, transport_costs,
                 price, sales]):
                print("Error: Please enter non-negative values.")
                continue

            new_data = pd.DataFrame([[  #create new dataframe
                population,
                inventory_levels,
                transport_costs,
                price,
                sales
            ]], columns=[
                "Population",
                "Inventory Levels (tons)",
                "Transportation Costs",
                "Sales",
                "Price"
            ])
            #use trained model to predict promotion outcome
            prediction = dt_model2.predict(new_data)[0]
            print("Prediction:", prediction)
            # Interpret prediction result
            if prediction == 1:
                print("Prediction: Cements most likely going to have a promotion.")
            else:
                print("Prediction: Cements not likely to have a promotion.")
            # Ask user if they want to run another prediction
            again = input("Enter another prediction? (yes/no): ")
            if again.lower() != "yes":
                break
       # Handle invalid inputs
        except ValueError:
            print("Invalid input. Please enter numbers.")
        except Exception as e:
            print("Model error:", e)
            print("Expected order:", dt_model2.feature_names_in_)
            break

predict_cement_type() # run the prediction function

In [ ]:
# Install Packages Needed to Visualize the Decision Tree
!pip install graphviz
!pip install pydotplus
!pip install six

In [ ]:
# Generate Decision Tree Visualization
from sklearn.tree import export_graphviz
from six import StringIO
from IPython.display import Image
import pydotplus

# Create DOT data for the tree
dot_data = StringIO()

# Export the trained tree into DOT format
export_graphviz(dt_model2,
                out_file=dot_data,
                filled=True,
                rounded=True,
                special_characters=True,
                class_names=['0', '1'])

# Convert DOT file into a graph
graph = pydotplus.graph_from_dot_data(dot_data.getvalue())

# Save the decision tree as an image
graph.write_png('cement.png')

# Display the decision tree
Image(graph.create_png())

**Decision Tree Lab Report**

A Decision Tree classification model was developed using the cement dataset to predict whether a cement product is likely to have a promotion based on five variables: population, inventory levels, transportation costs, sales, and price. The dataset was split into 70% training data and 30% testing data to evaluate the model’s performance. After training the model, it achieved an accuracy of approximately 50%, with precision, recall, and F1-score values also around 0.50, indicating that the model currently performs only slightly better than random prediction. From the decision tree visualization, inventory levels appear to be the most influential variable since it is used at the root of the tree to make the first decision split. For example, one node in the tree evaluates whether inventory levels are less than or equal to approximately 37,006 tons. This node contains 700 samples, with a class distribution of 356 observations without promotion and 344 with promotion, producing a Gini impurity of 0.50, which indicates the data at that point is highly mixed. The node predicts class 0 (no promotion) because it has slightly more observations. Overall, the model provides some initial insight into the relationship between inventory levels and promotion decisions, but additional tuning, more features, or a more advanced model would likely improve predictive performance.